In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname



root_path = dirname(os.getcwd()) + "/AdaTest"

pd.set_option("display.max_columns", None)
# data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/comuzzi/_processed/"
data_dir_graphs = root_path + "/data/datasets/comuzzi/graphs_repair/"

print(root_path, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = "cpu"

/home/sebastiano.dissegna/AdaTest
/home/sebastiano.dissegna/AdaTest/data/datasets/comuzzi/_processed/
/home/sebastiano.dissegna/AdaTest/data/datasets/comuzzi/graphs_repair/


In [7]:
results_path = root_path + "/results/results_CAISE/"

In [8]:
with open("comuzzi_results.json", 'r') as file:
    comuzzi_results = json.load(file)

In [9]:
our_results = {}

In [10]:
for dataset in comuzzi_results:
    our_results[dataset] = {}
    for t in comuzzi_results[dataset].keys():
        our_results[dataset][t] = {}
        d = pd.read_csv(f"{results_path}{dataset}_AT_only/{t}_V2_RESULTS_END.csv")
        our_results[dataset][t]["Activity_acc"] = d["Activity_acc"].values
        our_results[dataset][t]["time:timestamp_mae"] = d["time:timestamp_mae"].values

In [12]:
import scipy.stats as stats

In [13]:
statistics = {}
for dataset in comuzzi_results:
    statistics[dataset] = {}
    for t in comuzzi_results[dataset].keys():
        statistics[dataset][t] = {}
        for task in ["Activity_acc", "time:timestamp_mae"]:
            t_stat, p_value = stats.ttest_1samp(our_results[dataset][t][task], popmean=comuzzi_results[dataset][t][task])
            statistics[dataset][t][task] = {
                "t_statistic" : t_stat,
                "p_value": p_value
            }

/home/sebastiano.dissegna/AdaTest/v_env/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [14]:
statistics

{'bpi_2012_CZ': {'odd': {'Activity_acc': {'t_statistic': -14028.497079089013,
    'p_value': 2.419749648592291e-34},
   'time:timestamp_mae': {'t_statistic': 8.397078034325737,
    'p_value': 1.5000673826985686e-05}},
  'even': {'Activity_acc': {'t_statistic': -24962.3718541766,
    'p_value': 1.3530011632135064e-36},
   'time:timestamp_mae': {'t_statistic': 5.463845412285257,
    'p_value': 0.0003984307981705967}},
  'window': {'Activity_acc': {'t_statistic': -17079.568539772026,
    'p_value': 4.116979623505784e-35},
   'time:timestamp_mae': {'t_statistic': 6.337544124659197,
    'p_value': 0.00013484693452860168}},
  'random': {'Activity_acc': {'t_statistic': -18754.01617721054,
    'p_value': 1.7742802126938262e-35},
   'time:timestamp_mae': {'t_statistic': 15.183415933614864,
    'p_value': 1.0150489605564975e-07}},
  'attr_level': {'Activity_acc': {'t_statistic': -18779.290273423838,
    'p_value': 1.7529043210545272e-35},
   'time:timestamp_mae': {'t_statistic': 16.6386891773100

In [15]:
with open("Statistics_results.json", "w") as out:
    json.dump(statistics, out)